# Feature Engineering — Heart Attack Prediction

Este notebook transforma el dataset limpio en una representación numérica lista para ser consumida por los modelos. El proceso incluye la codificación del target, la binarización de variables Yes/No, la asignación de escalas ordinales con criterio clínico, el mapeo manual de categorías complejas y la aplicación de One-Hot Encoding para variables nominales. Al finalizar, se exporta un único archivo `dataset_features.csv` que centralizará la entrada para todos los notebooks de modelado.

## 0. Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Carga del dataset limpio

Se parte del archivo `heart_clean.csv` generado en el notebook de limpieza. Conviene verificar la forma del dataframe y la distribución del target antes de cualquier transformación, para detectar posibles inconsistencias introducidas durante el guardado.

In [ ]:
# ─── AJUSTA ESTA RUTA ───────────────────────────────────────
df = pd.read_csv('heart_clean.csv')
# ────────────────────────────────────────────────────────────

print(f'Shape: {df.shape}')
print(f'\nDistribución del target:')
print(df['HadHeartAttack'].value_counts())
print(f'\n% positivos: {df["HadHeartAttack"].value_counts(normalize=True)["Yes"]*100:.2f}%')

Shape: (413460, 40)

Distribución del target:
HadHeartAttack
No     389886
Yes     23574
Name: count, dtype: int64

% positivos: 5.70%


## 2. Clasificación de variables

Antes de codificar, se separan las columnas según el tipo de tratamiento que recibirán. Las variables binarias (Yes/No) se mapean directamente a 0/1. Las ordinales se convierten a enteros respetando el orden natural de sus categorías. Las nominales sin orden inherente se tratan con One-Hot Encoding. Adicionalmente, `BMI` se identifica como candidata a transformación logarítmica dado su sesgo a la derecha; la columna original se conserva para que cada modelo decida qué versión utilizar.

In [ ]:
TARGET = 'HadHeartAttack'

# Columnas que se benefician de log1p (distribución sesgada a la derecha)
# Las originales se conservan — cada notebook de modelo decide cuáles usar
LOG_COLS = ['BMI']

BINARY_COLS = [
    'PhysicalActivities', 'HadAngina', 'HadStroke', 'HadAsthma',
    'HadSkinCancer', 'HadCOPD', 'HadDepressiveDisorder', 'HadKidneyDisease',
    'HadArthritis', 'DeafOrHardOfHearing', 'BlindOrVisionDifficulty',
    'DifficultyConcentrating', 'DifficultyWalking', 'DifficultyDressingBathing',
    'DifficultyErrands', 'ChestScan', 'AlcoholDrinkers', 'HIVTesting',
    'FluVaxLast12', 'PneumoVaxEver', 'HighRiskLastYear'
]

# (columna, orden de menor a mayor)
ORDINAL_COLS = [
    ('GeneralHealth', ['Poor', 'Fair', 'Good', 'Very good', 'Excellent']),
    ('AgeCategory', [
        'Age 18 to 24', 'Age 25 to 29', 'Age 30 to 34', 'Age 35 to 39',
        'Age 40 to 44', 'Age 45 to 49', 'Age 50 to 54', 'Age 55 to 59',
        'Age 60 to 64', 'Age 65 to 69', 'Age 70 to 74', 'Age 75 to 79',
        'Age 80 or older'
    ]),
    ('LastCheckupTime', [
        '5 or more years ago',
        'Within past 5 years (2 years but less than 5 years ago)',
        'Within past 2 years (1 year but less than 2 years ago)',
        'Within past year (anytime less than 12 months ago)'
    ]),
    ('RemovedTeeth', ['None of them', '1 to 5', '6 or more, but not all', 'All']),
]

# Nominales → One-Hot Encoding
NOMINAL_COLS = ['State', 'RaceEthnicityCategory']

print('Columnas definidas.')

Columnas definidas.


## 3. Copia de trabajo y codificación del target

Se trabaja sobre una copia del dataframe para preservar el original en memoria. La variable objetivo `HadHeartAttack` se convierte a binario numérico: `Yes → 1`, `No → 0`.

In [ ]:
df_fe = df.copy()

df_fe[TARGET] = df_fe[TARGET].map({'Yes': 1, 'No': 0})

print('Target:')
print(df_fe[TARGET].value_counts())


Target:
HadHeartAttack
0    389886
1     23574
Name: count, dtype: int64


## 4. Variables binarias (Yes/No → 1/0)

Todas las variables con dominio `{Yes, No}` se mapean a `{1, 0}`. La variable `Sex` recibe un tratamiento análogo (`Female → 0`, `Male → 1`). Este paso es necesario para que los modelos puedan operar con estas columnas como entradas numéricas.

In [ ]:
for col in BINARY_COLS:
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].map({'Yes': 1, 'No': 0})

df_fe['Sex'] = df_fe['Sex'].map({'Female': 0, 'Male': 1})

print('Binarias codificadas ✓')


Binarias codificadas ✓


## 5. Variables ordinales

Las variables con categorías que siguen un orden lógico se codifican asignando un entero creciente a cada nivel. El orden se define explícitamente —de menor a mayor— para variables como `GeneralHealth`, `AgeCategory`, `LastCheckupTime` y `RemovedTeeth`. Este enfoque preserva la información de orden sin generar columnas adicionales, a diferencia del OHE.

In [ ]:
for col, order in ORDINAL_COLS:
    if col in df_fe.columns:
        present = [cat for cat in order if cat in df_fe[col].dropna().unique()]
        mapping = {cat: i for i, cat in enumerate(present)}
        df_fe[col] = df_fe[col].map(mapping)
        print(f'{col}: {mapping}')

print('\nOrdinales codificadas ✓')


GeneralHealth: {'Poor': 0, 'Fair': 1, 'Good': 2, 'Very good': 3, 'Excellent': 4}
AgeCategory: {'Age 18 to 24': 0, 'Age 25 to 29': 1, 'Age 30 to 34': 2, 'Age 35 to 39': 3, 'Age 40 to 44': 4, 'Age 45 to 49': 5, 'Age 50 to 54': 6, 'Age 55 to 59': 7, 'Age 60 to 64': 8, 'Age 65 to 69': 9, 'Age 70 to 74': 10, 'Age 75 to 79': 11, 'Age 80 or older': 12}
LastCheckupTime: {'5 or more years ago': 0, 'Within past 5 years (2 years but less than 5 years ago)': 1, 'Within past 2 years (1 year but less than 2 years ago)': 2, 'Within past year (anytime less than 12 months ago)': 3}
RemovedTeeth: {'None of them': 0, '1 to 5': 1, '6 or more, but not all': 2, 'All': 3}

Ordinales codificadas ✓


## 6. Variables multi-categoría con criterio clínico

Algunas variables no encajan en un mapeo automático porque sus categorías requieren interpretación clínica. Se codifican manualmente:

- **`HadDiabetes`**: se ordena por severidad diagnóstica (sin diabetes → prediabetes / gestacional → diabetes declarada).
- **`SmokerStatus`** y **`ECigaretteUsage`**: se ordenan por intensidad de consumo.
- **`TetanusLast10Tdap`**: se ordena por completitud del esquema de vacunación.
- **`CovidPos`**: se trata como binaria, agrupando el resultado positivo por test en casa junto al positivo confirmado.

In [ ]:
# HadDiabetes: ordinal por severidad
df_fe['HadDiabetes'] = df_fe['HadDiabetes'].map({
    'No': 0,
    'No, pre-diabetes or borderline diabetes': 1,
    'Yes, but only during pregnancy (female)': 1,
    'Yes': 2
})

# SmokerStatus: ordinal por intensidad de consumo
df_fe['SmokerStatus'] = df_fe['SmokerStatus'].map({
    'Never smoked': 0,
    'Former smoker': 1,
    'Current smoker - now smokes some days': 2,
    'Current smoker - now smokes every day': 3
})

# ECigaretteUsage: ordinal por frecuencia
df_fe['ECigaretteUsage'] = df_fe['ECigaretteUsage'].map({
    'Never used e-cigarettes in my entire life': 0,
    'Not at all (right now)': 1,
    'Use them some days': 2,
    'Use them every day': 3
})

# TetanusLast10Tdap: ordinal por completitud de vacunación
df_fe['TetanusLast10Tdap'] = df_fe['TetanusLast10Tdap'].map({
    'No, did not receive any tetanus shot in the past 10 years': 0,
    'Yes, received tetanus shot but not sure what type': 1,
    'Yes, received tetanus shot, but not Tdap': 2,
    'Yes, received Tdap': 3
})

# CovidPos: binaria (positivo o no)
df_fe['CovidPos'] = df_fe['CovidPos'].map({
    'No': 0,
    'Yes': 1,
    'Tested positive using home test without a health professional': 1
})

print('Multi-categorías codificadas ✓')


Multi-categorías codificadas ✓


## 7. Variables nominales — One-Hot Encoding

Las variables `State` y `RaceEthnicityCategory` no tienen un orden natural, por lo que se expanden mediante OHE. Se descarta la primera categoría de cada variable (`drop_first=True`) para evitar multicolinealidad perfecta. Las columnas resultantes son de tipo entero.

In [ ]:
nominal_presentes = [c for c in NOMINAL_COLS if c in df_fe.columns]

df_fe = pd.get_dummies(df_fe, columns=nominal_presentes, drop_first=True, dtype=int)

print(f'Shape tras OHE: {df_fe.shape}')
ohe_cols = [c for c in df_fe.columns if any(c.startswith(n + '_') for n in nominal_presentes)]
print(f'Columnas OHE generadas: {len(ohe_cols)}')


Shape tras OHE: (413460, 95)
Columnas OHE generadas: 57


## 8. Verificación de nulos residuales

Tras todas las transformaciones se comprueba que no hayan quedado nulos. Si el mapeo de alguna categoría imprevista produjo `NaN`, se imputa con la mediana de la columna correspondiente.

In [ ]:
nulos = df_fe.isnull().sum()
nulos_presentes = nulos[nulos > 0]

if len(nulos_presentes) == 0:
    print('✅ Sin nulos.')
else:
    print(f'⚠️  Nulos residuales en {len(nulos_presentes)} columnas:')
    print(nulos_presentes)
    print('\nImputando con mediana...')
    for col in nulos_presentes.index:
        df_fe[col] = df_fe[col].fillna(df_fe[col].median())
    print('✅ Imputación completada.')

✅ Sin nulos.


## 9. Resumen del dataset final

Se imprime un resumen con la forma del dataset, la distribución del target y los tipos de datos resultantes, como confirmación de que el pipeline de feature engineering se ejecutó correctamente.

In [ ]:
print('='*60)
print('DATASET FINAL')
print('='*60)
print(f'Shape: {df_fe.shape}')
print(f'Target (1=HeartAttack): {df_fe[TARGET].value_counts().to_dict()}')
print(f'% positivos: {df_fe[TARGET].mean()*100:.2f}%')
print(f'\nColumnas log disponibles: {[c for c in df_fe.columns if c.endswith("_log")]}')
print(f'Feature de interacción BMI_x_Age: {"BMI_x_Age" in df_fe.columns}')
print(f'\nTipos de datos:')
print(df_fe.dtypes.value_counts())
print('\nPrimeras filas:')
df_fe.head(3)

DATASET FINAL
Shape: (413460, 95)
Target (1=HeartAttack): {0: 389886, 1: 23574}
% positivos: 5.70%

Columnas log disponibles: []
Feature de interacción BMI_x_Age: False

Tipos de datos:
int64      89
float64     6
Name: count, dtype: int64

Primeras filas:


,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,HadAngina,HadStroke,HadAsthma,HadSkinCancer,HadCOPD,HadDepressiveDisorder,HadKidneyDisease,HadArthritis,HadDiabetes,DeafOrHardOfHearing,BlindOrVisionDifficulty,DifficultyConcentrating,DifficultyWalking,DifficultyDressingBathing,DifficultyErrands,SmokerStatus,ECigaretteUsage,ChestScan,AgeCategory,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos,State_Alaska,State_Arizona,State_Arkansas,State_California,State_Colorado,State_Connecticut,State_Delaware,State_District of Columbia,State_Florida,State_Georgia,State_Guam,State_Hawaii,State_Idaho,State_Illinois,State_Indiana,State_Iowa,State_Kansas,State_Kentucky,State_Louisiana,State_Maine,State_Maryland,State_Massachusetts,State_Michigan,State_Minnesota,State_Mississippi,State_Missouri,State_Montana,State_Nebraska,State_Nevada,State_New Hampshire,State_New Jersey,State_New Mexico,State_New York,State_North Carolina,State_North Dakota,State_Ohio,State_Oklahoma,State_Oregon,State_Pennsylvania,State_Puerto Rico,State_Rhode Island,State_South Carolina,State_South Dakota,State_Tennessee,State_Texas,State_Utah,State_Vermont,State_Virgin Islands,State_Virginia,State_Washington,State_West Virginia,State_Wisconsin,State_Wyoming,RaceEthnicityCategory_Hispanic,"RaceEthnicityCategory_Multiracial, Non-Hispanic","RaceEthnicityCategory_Other race only, Non-Hispanic","RaceEthnicityCategory_White only, Non-Hispanic"
0,0,3,0.0000,0.0000,3,0,8.0000,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,1,0,12,1.7000,80.7400,27.9377,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,0,4,0.0000,0.0000,3,0,6.0000,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,12,1.6000,68.0400,26.5781,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,0,3,2.0000,3.0000,3,1,5.0000,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,1.5700,63.5000,25.7617,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


## 10. Exportación

El dataset procesado se guarda como `dataset_features.csv`. Este archivo será la entrada estándar para todos los notebooks de modelado, garantizando que todos los modelos partan exactamente del mismo conjunto de features.

In [ ]:
OUTPUT_PATH = 'dataset_features.csv'

df_fe.to_csv(OUTPUT_PATH, index=False)

print(f'✅ Dataset exportado → {OUTPUT_PATH}')
print(f'   {df_fe.shape[0]:,} filas × {df_fe.shape[1]} columnas')



✅ Dataset exportado → dataset_features.csv
   413,460 filas × 95 columnas
